# IA101 — Notebook 6 — Application intégratrice et synthèse

**Durée : 20 minutes** · Objectifs visés : **O3, O4, O5** — démarche complète, en autonomie

Ce notebook mobilise l'ensemble de la démarche sur un problème nouveau, avec une consigne ouverte.

### Le problème
Le jeu de données `load_diabetes` décrit **442 patients diabétiques** par 10 variables : âge, sexe, indice de masse corporelle (IMC), tension artérielle et 6 mesures sanguines. La cible est un **indicateur de progression de la maladie un an plus tard** (valeurs de 25 à 346).

Il s'agit d'un problème de **régression**.

**Consigne :** construire le modèle qui obtient la plus petite **erreur absolue moyenne (EAM)** sur l'ensemble de test.

$$\text{EAM} = \frac{1}{n} \sum_{i=1}^{n} \lvert \hat{y}_i - y_i \rvert$$

L'EAM s'exprime dans l'unité de la cible, ce qui facilite son interprétation.

### Modalités
- Travail en équipe de deux dans les salles de discussion, ou individuel.
- Le découpage fourni ci-dessous est imposé (même `random_state` pour toutes les équipes).
- L'ensemble de test n'est utilisé **qu'une fois**, pour le résultat final.
- Chaque équipe inscrit son EAM finale dans le clavardage. L'équipe ayant le meilleur résultat présente sa démarche en une minute.

In [1]:
# Outil de questions de vérification (exécutez cette cellule sans la modifier)
import ipywidgets as widgets
from IPython.display import display, Markdown

def quiz(question, options, bonne, explication):
    """Affiche une question à choix multiples avec rétroaction immédiate."""
    choix = widgets.RadioButtons(options=options, value=None,
                                 layout={"width": "max-content"})
    bouton = widgets.Button(description="Vérifier", button_style="primary")
    sortie = widgets.Output()

    def verifier(_):
        with sortie:
            sortie.clear_output()
            if choix.value is None:
                display(Markdown("Veuillez choisir une réponse."))
            elif choix.value == options[bonne]:
                display(Markdown(f"**Bonne réponse.** {explication}"))
            else:
                display(Markdown("**Réponse incorrecte.** Relisez la section précédente et essayez de nouveau."))

    bouton.on_click(verifier)
    display(Markdown(f"**Question de vérification.** {question}"), choix, bouton, sortie)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

diabete = load_diabetes(scaled=False)
X = pd.DataFrame(diabete.data, columns=["âge", "sexe", "IMC", "tension", "s1", "s2", "s3", "s4", "s5", "s6"])
y = diabete.target

# Découpage imposé : ne pas modifier
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
X.head()

,âge,sexe,IMC,tension,s1,s2,s3,s4,s5,s6
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0


### Démarche recommandée

| Étape | Question | Référence |
| --- | --- | --- |
| 1. Explorer | Quelles variables ? Quelles échelles ? Quelles relations avec la cible ? | Notebooks 2 et 5 |
| 2. Établir une référence | Quelle erreur obtient un modèle trivial ? Un modèle simple ? | Notebooks 1 et 3 |
| 3. Comparer | Quel modèle est le meilleur en validation croisée ? | Notebook 4 |
| 4. Régler | Quels hyperparamètres l'améliorent ? | Notebook 4 |
| 5. Évaluer | Quelle est l'EAM finale sur le test ? | Notebooks 3 et 4 |

Pour mesurer l'EAM avec `cross_val_score` et `GridSearchCV`, utilisez `scoring="neg_mean_absolute_error"`. scikit-learn maximise toujours le score : il renvoie donc l'EAM **avec un signe négatif**.

### Étape 1 — Explorer

In [ ]:
# Indice : X.describe(), corrélations avec y, nuages de points
...

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
display(X.describe().round(2).T[["mean", "std", "min", "max"]])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, var in zip(axes, ["IMC", "s5", "âge"]):
    ax.scatter(X[var], y, s=10, alpha=0.6)
    ax.set_xlabel(var); ax.set_ylabel("progression")
    ax.set_title(f"corrélation = {np.corrcoef(X[var], y)[0, 1]:.2f}")
plt.tight_layout(); plt.show()
# Échelles très différentes (âge ~ 50, s1 ~ 190) : normalisation nécessaire pour le KNN et Ridge.
# L'IMC et s5 sont les variables les plus corrélées à la progression.
```

</details>

### Étape 2 — Établir une référence

In [ ]:
# Indice : DummyRegressor (prédit la moyenne), puis LinearRegression, en validation croisée
...

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
for nom, m in {"Référence (moyenne)": DummyRegressor(), "Régression linéaire": LinearRegression()}.items():
    eam = -cross_val_score(m, X_ent, y_ent, cv=5, scoring="neg_mean_absolute_error").mean()
    print(f"{nom:<22} EAM = {eam:.1f}")
```

</details>

### Étape 3 — Comparer plusieurs modèles

In [ ]:
candidats = {
    # À COMPLÉTER : au moins trois modèles (pensez aux Pipeline avec StandardScaler)
}
...

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
candidats = {
    "Régression linéaire": LinearRegression(),
    "Ridge (normalisé)": make_pipeline(StandardScaler(), Ridge()),
    "KNN (normalisé)": make_pipeline(StandardScaler(), KNeighborsRegressor()),
    "Forêt aléatoire": RandomForestRegressor(n_estimators=300, random_state=0),
}
for nom, m in candidats.items():
    scores = -cross_val_score(m, X_ent, y_ent, cv=5, scoring="neg_mean_absolute_error")
    print(f"{nom:<22} EAM = {scores.mean():.1f} ± {scores.std():.1f}")
```

</details>

### Étape 4 — Régler les hyperparamètres

In [ ]:
# Indice : GridSearchCV(pipeline, grille, cv=5, scoring="neg_mean_absolute_error")
...

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
recherche_knn = GridSearchCV(
    make_pipeline(StandardScaler(), KNeighborsRegressor()),
    {"kneighborsregressor__n_neighbors": [5, 10, 15, 20, 30, 40],
     "kneighborsregressor__weights": ["uniform", "distance"]},
    cv=5, scoring="neg_mean_absolute_error").fit(X_ent, y_ent)
print("KNN   :", recherche_knn.best_params_, f"| EAM en validation = {-recherche_knn.best_score_:.1f}")

recherche_ridge = GridSearchCV(
    make_pipeline(StandardScaler(), Ridge()),
    {"ridge__alpha": [0.01, 0.1, 1, 3, 10, 30, 100]},
    cv=5, scoring="neg_mean_absolute_error").fit(X_ent, y_ent)
print("Ridge :", recherche_ridge.best_params_, f"| EAM en validation = {-recherche_ridge.best_score_:.1f}")

meilleur = min([recherche_knn, recherche_ridge], key=lambda r: -r.best_score_).best_estimator_
print("Modèle retenu :", meilleur)
```

</details>

### Étape 5 — Évaluation finale (une seule fois)

In [ ]:
# Calculez l'EAM du modèle retenu sur l'ensemble de test
...

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
eam_finale = mean_absolute_error(y_test, meilleur.predict(X_test))
print(f"EAM finale sur l'ensemble de test : {eam_finale:.1f}")
print(f"Pour comparaison : la cible varie de {y.min():.0f} à {y.max():.0f}, écart-type {y.std():.0f}")

plt.figure(figsize=(4.8, 4.5))
plt.scatter(y_test, meilleur.predict(X_test), alpha=0.7)
plt.plot([25, 346], [25, 346], "k--", label="Prédiction parfaite")
plt.xlabel("Progression observée"); plt.ylabel("Progression prédite")
plt.title("Prédictions sur l'ensemble de test"); plt.legend(); plt.show()
```

</details>

**Analyse.** L'EAM de test est-elle cohérente avec l'EAM estimée en validation croisée ? Pourquoi ce problème est-il plus difficile que la classification des iris ?

<details>
<summary><b>Éléments de réponse</b></summary>

- Une EAM de test proche de l'estimation par validation croisée confirme que l'évaluation n'est pas biaisée.
- Les 10 variables ne contiennent qu'une partie de l'information qui détermine la progression (traitement, mode de vie, génétique). Aucun modèle ne peut prédire une information absente des données : le levier principal serait de collecter d'autres variables.
</details>

---

## Synthèse de l'atelier : test de positionnement final

Les mêmes questions qu'au début de l'atelier. Comparez avec vos réponses du notebook 0.

In [ ]:
quiz("1. Que « apprend » un modèle pendant l'entraînement ?",
     ["Les réponses par cœur", "Les valeurs de ses paramètres qui minimisent la perte", "Ses hyperparamètres"],
     1, "Apprendre consiste à estimer les paramètres qui minimisent la fonction de perte.")

In [ ]:
quiz("2. À quoi sert l'ensemble de test ?",
     ["À entraîner le modèle", "À choisir les hyperparamètres", "À estimer la performance sur des données non vues"],
     2, "Il n'est utilisé qu'une fois, pour l'évaluation finale.")

In [ ]:
quiz("3. Exactitude d'entraînement : 99 %. Exactitude de test : 70 %. Diagnostic ?",
     ["Surapprentissage", "Sous-apprentissage", "Modèle bien ajusté"],
     0, "Le modèle ajuste le bruit des données d'entraînement et ne généralise pas.")

In [ ]:
quiz("4. Pour un test de dépistage, quelle métrique faut-il surveiller en priorité ?",
     ["L'exactitude", "La précision", "Le rappel"],
     2, "Le rappel mesure la proportion de cas positifs détectés.")

In [ ]:
quiz("5. Laquelle de ces pratiques fausse l'évaluation d'un modèle ?",
     ["Normaliser les variables dans un Pipeline", "Choisir les hyperparamètres en consultant l'ensemble de test", "Utiliser la validation croisée"],
     1, "Le choix des réglages doit se faire par validation croisée sur l'entraînement uniquement.")

### Synthèse des objectifs

| Objectif | Ce que vous avez réalisé |
| --- | --- |
| O1 — Situer un problème | Classification, régression et regroupement (notebook 0) |
| O2 — Expliquer l'apprentissage | Perte, gradient dérivé et vérifié, descente de gradient, généralisation (notebook 1) |
| O3 — Entraîner un modèle | KNN, régression logistique, forêts, Ridge avec l'interface `fit` / `predict` / `score` (notebooks 2 et 6) |
| O4 — Évaluer | Matrice de confusion, précision, rappel, seuil, EAM (notebooks 3 et 6) |
| O5 — Améliorer | Normalisation, `Pipeline`, validation croisée, `GridSearchCV`, courbes de validation et d'apprentissage (notebooks 2, 4 et 6) |
| O6 — Détecter des structures | k-moyennes, silhouette, ACP (notebook 5) |

### Pour aller plus loin
- **Atelier IA102 — Apprentissage profond avec PyTorch** : lorsque les données sont brutes (images, textes) et abondantes, les réseaux de neurones prennent le relais.
- [Guide de l'utilisateur de scikit-learn](https://scikit-learn.org/stable/user_guide.html)
- [Documentation technique de l'Alliance](https://docs.alliancecan.ca)
- L'aide-mémoire IA101, fourni avec les notebooks.